# URECA: Random Forest Prediction of Fluorophore Photophysical Properties

**Project:** NTU URECA (self-directed), supervised by Prof Liu

This is a single notebook covering the whole pipeline: setup, EDA, featurization, modeling, analysis, and write-up. Everything is built in order, in this one file.

---

## Goal

We're predicting four photophysical properties of organic fluorophores (chromophores) from molecular structure (SMILES string):

| Target | Column in raw data | Notes |
|---|---|---|
| Absorption max | `Absorption max (nm)` | |
| Emission max | `Emission max (nm)` | |
| Extinction coefficient | `log(e/mol-1 dm3 cm-1)` | already log10-scaled in the raw data |
| Quantum yield (Φ_F) | `Quantum yield` | bounded 0–1, expected to be the hardest target |

**Model:** Random Forest, as the first model.

**Key method:** train/test split by **Murcko scaffold** (via RDKit), not randomly. A random split can let near-identical molecules appear on both sides, so the model can do well just by memorizing similar structures. Comparing performance on a scaffold split vs. a random split shows how well the model actually generalizes to new chemical structures — this gap is a potential result worth reporting.

## Notebook sections

1. **Setup & data loading**
2. EDA — distributions, missingness, solvent effects, correlations
3. Featurization + scaffold split
4. RF tuning
5. Feature importance + scaffold-vs-random gap analysis
6. Error analysis
7. Write-up

## Dataset

`chromophore_data.csv`, in this same folder. 20,836 rows, one row per (chromophore, solvent) measurement. Columns: SMILES, solvent, the four target properties, plus lifetime, FWHM, molecular weight, and reference.

Two things to note: not every row has all four targets (missingness varies by target), and the same molecule can appear multiple times under different solvents. Both are covered in the EDA section below.

## 1. Imports

`pandas`/`numpy` for data handling, `matplotlib`/`seaborn` for plots, `rdkit` for chemistry (reading SMILES, scaffolds, descriptors), `scikit-learn` for the model and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # RDKit likes to complain about every weird SMILES - silence that for now

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

print('rdkit, sklearn, pandas all imported OK')

## 2. Load the raw dataset

Read the CSV from this folder. `df_raw` stays untouched as a backup; all work happens on `df`.

In [ ]:
DATA_PATH = 'chromophore_data.csv'

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
df = df_raw.copy()  # work on a copy, keep df_raw untouched as a backup

print(f'Loaded {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 3. Rename columns

The raw column names (e.g. `Absorption max (nm)`) are long to type repeatedly, so we rename them to short codes. `df_raw` keeps the original names.

In [ ]:
RENAME_MAP = {
    'Chromophore': 'smiles',
    'Solvent': 'solvent',
    'Absorption max (nm)': 'abs_max',
    'Emission max (nm)': 'emi_max',
    'Quantum yield': 'qy',
    'log(e/mol-1 dm3 cm-1)': 'log_ext_coeff',
    'Molecular weight (g mol-1)': 'mol_weight',
}

df = df.rename(columns=RENAME_MAP)

# these two lists get reused constantly from here on, so let's name them once
TARGET_COLS = ['abs_max', 'emi_max', 'log_ext_coeff', 'qy']
ID_COLS = ['smiles', 'solvent']

df[ID_COLS + TARGET_COLS].head()

## 4. Sanity checks

Three checks before doing anything else: how much data each target actually has, whether RDKit can parse every SMILES string, and how many unique molecules we have (since the same molecule appears under multiple solvents).

In [ ]:
print('Non-null counts per target (out of {:,} rows):'.format(len(df)))
print(df[TARGET_COLS].notna().sum())
print()
print('Unique SMILES (molecules):', df['smiles'].nunique())
print('Unique solvents:', df['solvent'].nunique())

In [ ]:
def parses_ok(smi):
    # if RDKit can't parse a SMILES, Chem.MolFromSmiles just quietly returns None
    try:
        return Chem.MolFromSmiles(smi) is not None
    except Exception:
        return False

parse_mask = df['smiles'].apply(parses_ok)
print(f'SMILES that RDKit can parse: {parse_mask.sum():,} / {len(df):,} ({parse_mask.mean():.1%})')

if (~parse_mask).any():
    print('\nExample unparseable SMILES:')
    display(df.loc[~parse_mask, ['smiles']].head())

## 5. Exploratory Data Analysis

Go through the four targets one at a time: distribution, missingness, outliers, then a short findings note before moving to the next. After all four, look at cross-target relationships (e.g. Stokes shift = emission − absorption) and solvent effects.

### 5.1 Absorption max (nm)

In [ ]:
# how many rows actually have this value, and what does it look like on average?
print(f"Missing: {df['abs_max'].isna().sum():,} / {len(df):,} rows ({df['abs_max'].isna().mean():.1%})")
df['abs_max'].describe()

In [ ]:
# histogram for the overall shape, boxplot to spot outliers at a glance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['abs_max'].dropna(), bins=60, ax=axes[0], color='steelblue')
axes[0].set_title('Absorption max — distribution')
axes[0].set_xlabel('Absorption max (nm)')

sns.boxplot(x=df['abs_max'].dropna(), ax=axes[1], color='steelblue')
axes[1].set_title('Absorption max — boxplot (outlier check)')
axes[1].set_xlabel('Absorption max (nm)')

plt.tight_layout()
plt.show()

In [ ]:
# real UV-Vis-NIR absorption lives roughly in the 200-1000 nm window,
# so anything outside that is worth eyeballing - could be a typo in the source paper
low = df[df['abs_max'] < 200][['smiles', 'solvent', 'abs_max']]
high = df[df['abs_max'] > 1000][['smiles', 'solvent', 'abs_max']]

print(f'Below 200 nm: {len(low)} rows')
print(f'Above 1000 nm: {len(high)} rows')
display(pd.concat([low, high]))

**Findings — Absorption max:**

- **Coverage:** 17,703 / 20,836 rows (~85%) — the best-covered target.
- **Shape:** 127.7–1055.0 nm, median 397 nm, right-skewed (skew ≈ 1.25). Most values fall in the near-UV/visible range (350–480 nm is the middle 50%), with a long tail toward the NIR.
- **Outliers checked:** 21 rows below 200 nm and 1 above 1000 nm. All are explainable — the sub-200nm rows are small gas-phase molecules (methane, ethane, propane, etc.) that genuinely absorb in the far-UV; the >1000nm row is a large conjugated dye. None are data errors, so nothing gets dropped.
- **No other issues** — no zeros or negatives.
- **For modeling:** Random Forest handles skew and outliers fine, so no transform is needed for this target.

Next: Emission max.

### 5.2 Emission max (nm)

Same checks as above, plus one extra: emission should physically occur at a longer wavelength than absorption (Stokes shift), so we can use that as a sanity check.

In [ ]:
print(f"Missing: {df['emi_max'].isna().sum():,} / {len(df):,} rows ({df['emi_max'].isna().mean():.1%})")
df['emi_max'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['emi_max'].dropna(), bins=60, ax=axes[0], color='darkorange')
axes[0].set_title('Emission max — distribution')
axes[0].set_xlabel('Emission max (nm)')

sns.boxplot(x=df['emi_max'].dropna(), ax=axes[1], color='darkorange')
axes[1].set_title('Emission max — boxplot (outlier check)')
axes[1].set_xlabel('Emission max (nm)')

plt.tight_layout()
plt.show()

In [ ]:
# physics check: a molecule loses a bit of energy before it emits, so emission
# wavelength should always be >= absorption wavelength (positive Stokes shift).
# if we see the opposite, that's either unusual photophysics or a data issue worth a look.
both = df.dropna(subset=['emi_max', 'abs_max']).copy()
both['stokes_shift'] = both['emi_max'] - both['abs_max']

neg_shift = both[both['stokes_shift'] < 0].sort_values('stokes_shift')
print(f"Rows with emission < absorption (negative Stokes shift): {len(neg_shift)} / {len(both)}")
display(neg_shift[['smiles', 'solvent', 'abs_max', 'emi_max', 'stokes_shift']].head(10))

**Findings — Emission max:**

- **Coverage:** 18,851 / 20,836 rows (~90%) — better covered than absorption.
- **Shape:** 247–1050 nm, median 488 nm, mildly right-skewed (skew ≈ 0.66, less skewed than absorption). Consistent with the Stokes shift pushing emission to longer wavelengths than absorption.
- **Outliers:** only 1 row above 1000 nm (a long conjugated dye, plausible), none below 200 nm.
- **Stokes shift check:** 17 / 16,422 rows (~0.1%) have emission at a shorter wavelength than absorption, which shouldn't normally happen. 15 are small (≤15 nm), likely measurement noise. 2 are larger (-57, -55 nm, both BODIPY-dimer structures in toluene) — worth a second look later, but too few rows to drop.
- **For modeling:** cleaner than absorption max, no transform needed.

Next: log(extinction coefficient).

### 5.3 log(extinction coefficient)

Already log10-scaled in the raw data. This target has the most missingness of the four, so coverage matters here more than for the others.

In [ ]:
print(f"Missing: {df['log_ext_coeff'].isna().sum():,} / {len(df):,} rows ({df['log_ext_coeff'].isna().mean():.1%})")
df['log_ext_coeff'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['log_ext_coeff'].dropna(), bins=60, ax=axes[0], color='seagreen')
axes[0].set_title('log(extinction coefficient) — distribution')
axes[0].set_xlabel('log10(ε / mol⁻¹ dm³ cm⁻¹)')

sns.boxplot(x=df['log_ext_coeff'].dropna(), ax=axes[1], color='seagreen')
axes[1].set_title('log(extinction coefficient) — boxplot (outlier check)')
axes[1].set_xlabel('log10(ε / mol⁻¹ dm³ cm⁻¹)')

plt.tight_layout()
plt.show()

In [ ]:
# real molar extinction coefficients for organic dyes top out around 10^5-10^6
# (log10 ~ 5-6). anything above that is a candidate data error, not real chemistry.
high = df[df['log_ext_coeff'] > 6][['smiles', 'solvent', 'log_ext_coeff', 'mol_weight', 'Reference']]
print(f'Rows above log10(ε) = 6: {len(high)}')
display(high)

In [ ]:
# two of those four rows (log10(e) = 10.4 and 9.0) are from the same paper and the
# same family of near-identical molecules, where every other row sits around 5.5-6.0.
# that's a strong sign those two specific values are data-entry errors, not real values.
# fixing it directly on df - df_raw still has the original numbers if we ever need them.
error_mask = df['log_ext_coeff'] > 7
print(f'Setting {error_mask.sum()} likely data-error rows to NaN in log_ext_coeff')
df.loc[error_mask, 'log_ext_coeff'] = np.nan

**Findings — log(extinction coefficient):**

- **Coverage:** only 8,300 / 20,836 rows (~40%) — by far the most missing of the four targets. Fewer papers report this than absorption/emission.
- **Shape:** 1.08–10.4 before cleaning, median 4.46, left-skewed (skew ≈ -1.40, the only target skewed in this direction). Most values sit between 4.1 and 4.7 (log10 scale), i.e. ε roughly 10⁴–10⁵.
- **Outliers:** 4 rows above log10(ε) = 6. Two (10.4 and 9.0) are from the same paper and the same family of near-identical molecules, where every comparable structure in that paper sits around 5.5–6.0 — these two are almost certainly data-entry errors, not real chemistry (a molar extinction coefficient of 10⁹–10¹⁰ is physically implausible for an organic dye). Set to NaN rather than left in. The other two (6.02, 6.11) are plausible for strongly absorbing dyes and were kept.
- **For modeling:** low coverage means this target will have the smallest usable training set of the four — worth keeping in mind when comparing scaffold-split performance across targets later, since a smaller training set makes the scaffold-split gap harder to measure precisely.

Next: Quantum yield.

### 5.4 Quantum yield (Φ_F)

Bounded between 0 and 1 by definition, so the main things to check are whether the data actually respects that bound, and what the shape looks like near the edges.

In [ ]:
print(f"Missing: {df['qy'].isna().sum():,} / {len(df):,} rows ({df['qy'].isna().mean():.1%})")
df['qy'].describe()

In [ ]:
# quantum yield is bounded [0,1] by definition, so confirm the data actually respects that
qy = df['qy']
print('Values < 0:', (qy < 0).sum())
print('Values > 1:', (qy > 1).sum())
print('Values == 0:', (qy == 0).sum())
print('Values < 0.01:', (qy < 0.01).sum())
print('Values > 0.99:', (qy > 0.99).sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['qy'].dropna(), bins=50, ax=axes[0], color='mediumorchid')
axes[0].set_title('Quantum yield — distribution')
axes[0].set_xlabel('Quantum yield (Φ_F)')

sns.boxplot(x=df['qy'].dropna(), ax=axes[1], color='mediumorchid')
axes[1].set_title('Quantum yield — boxplot')
axes[1].set_xlabel('Quantum yield (Φ_F)')

plt.tight_layout()
plt.show()

**Findings — Quantum yield:**

- **Coverage:** 13,978 / 20,836 rows (~67%).
- **All values are within [0, 1]** — no cleaning needed on that front.
- **Shape is the real issue:** median 0.27, mean 0.35, mildly right-skewed (skew ≈ 0.57), but the skew number undersells it. 150 rows are exactly 0, and 1,273 (~9%) are below 0.01 — a sharp spike of non-emissive or near-non-emissive molecules at the low end. There's a separate smaller pile near 1 (148 rows above 0.99). So the real shape is closer to a U/bimodal distribution with a big mass at zero, not a smooth skewed curve.
- **Why this is the hardest target:** a large zero-inflated spike like this is hard for any regressor, RF included, to fit well — it has to learn a sharp discontinuity (near-zero vs. everything else) rather than a smooth function of molecular features. This matches the expectation going in.
- **For modeling:** worth trying a couple of specific strategies when we get there — e.g. a classification step (emissive vs. non-emissive) before regression, or just accepting a weaker R² here than for the wavelength targets and reporting it as expected, not a bug.

That's all four targets. Summary review next.

### 5.5 Data cleaning & target strategy — decisions going into modeling

Summary of what we decided after reviewing all four targets, before moving on to featurization:

- **Model structure:** four separate Random Forest models, one per target. No multi-output model.
- **Absorption max, Emission max:** no transform. Skew is mild (1.25, 0.66) and all checked outliers turned out to be real chemistry, not errors. Train on raw nm values.
- **log(extinction coefficient):** already log10-scaled in the raw data. Two rows (10.4, 9.0) were corrected to NaN as data-entry errors (Section 5.3). No further transform on top of the existing log scale.
- **Quantum yield:** train the baseline RF on the raw [0,1] values, no transform, even though the distribution is zero-inflated and this target is expected to perform worse. This keeps the core deliverable simple (one RF per target, same recipe each time) and gives an honest baseline number to report.
  - **Follow-up experiment (later, time permitting):** once the baseline for all four targets is done, revisit quantum yield specifically with an arcsine-sqrt transform and/or a two-stage classify-then-regress approach, and compare against the baseline empirically rather than assuming it's better.
- **Outlier policy in general:** no capping/winsorizing. Random Forest isolates extreme points into their own leaves rather than letting them distort a global fit the way a linear model would, so outliers we've confirmed as real don't need special handling.
- **Derived quantities (e.g. Stokes shift) are for EDA/error-analysis only** — they use a second target as input, so they can't be used as model features without leaking the thing we're trying to predict.

Next: featurization (turning SMILES into model inputs) and the scaffold split.

## 6. Data Cleaning

One more structural issue to handle before featurization: the same (molecule, solvent) pair sometimes appears as more than one row, because different papers measured the same compound independently. Checking how common this is and what to do about it.

In [ ]:
# same (smiles, solvent) pair measured more than once, usually by different papers
dup_mask = df.duplicated(subset=['smiles', 'solvent'], keep=False)
n_dup_rows = dup_mask.sum()
n_dup_groups = df[dup_mask].groupby(['smiles', 'solvent']).ngroups

print(f'Rows involved in a duplicate (smiles, solvent) pair: {n_dup_rows} / {len(df)}')
print(f'Number of distinct duplicated groups: {n_dup_groups}')

# how much do the repeated measurements actually disagree with each other?
spread = (df[dup_mask].groupby(['smiles', 'solvent'])['abs_max']
          .agg(lambda x: x.dropna().max() - x.dropna().min() if x.notna().sum() > 1 else np.nan))
print(f"\nAbsorption max spread within duplicate groups (where measured twice):")
print(spread.dropna().describe())
print(f"Groups disagreeing by >10 nm: {(spread.dropna() > 10).sum()} / {spread.dropna().shape[0]}")

In [ ]:
# collapse duplicate (smiles, solvent) rows into one, taking the median of each target.
# median is robust to the occasional one-off disagreement between papers without needing
# to pick a "correct" paper. keep one reference string per group just for traceability.
agg_dict = {col: 'median' for col in TARGET_COLS}
agg_dict['mol_weight'] = 'median'
agg_dict['Reference'] = lambda refs: '; '.join(sorted(set(refs.dropna().astype(str))))

df = df.groupby(['smiles', 'solvent'], as_index=False).agg(agg_dict)

print(f'Rows before dedup: 20,836  ->  after dedup: {len(df):,}')
print(f'Unique molecules: {df["smiles"].nunique():,}')
print()
print('Non-null counts per target after dedup:')
print(df[TARGET_COLS].notna().sum())

**Cleaning summary:**

- **Duplicates:** 618 rows (~3%) shared a (smiles, solvent) pair with at least one other row, across 304 groups — mostly different papers remeasuring the same compound. Most groups agreed closely; 22 groups disagreed by more than 10 nm on absorption max. Collapsed via median per target, one row per (smiles, solvent) going forward. 20,836 → 20,522 rows; unique molecule count (6,865) is unchanged, since dedup only merges repeated solvent measurements, not distinct molecules.
- **log(extinction coefficient) errors:** already corrected to NaN in Section 5.3 (2 rows).
- **SMILES validity:** already confirmed 100% parseable in Section 4 — no rows dropped on that basis.
- **No other cleaning needed** — absorption max, emission max, and quantum yield all passed their checks in Section 5 as-is.

This `df` is now the cleaned working dataset. `df_raw` still holds the original, untouched 20,836-row file if we ever need to double check something against the source.

Next: featurization — turning each SMILES string into numeric inputs the Random Forest can use, and then the Murcko scaffold split.